# Decoder-Only Transformer (GPT)

A compact, character-level GPT in PyTorch on TinyShakespeare.

## Goals

1. Build core GPT layers.
2. Verify training behavior.
3. Generate text.

## Architecture

`tokens -> token+position embeddings -> Transformer blocks x N -> LayerNorm -> linear head`

Uses **pre-norm** blocks (`LayerNorm` before each sublayer).

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------
# Model hyperparameters
# ---------------------
CONFIG = {
    "seed": 42,
    "block_size": 128,
    "batch_size": 32,
    "n_embd": 128,
    "n_heads": 4,
    "n_layers": 4,
    "dropout": 0.2,
    "learning_rate": 3e-4,
    "weight_decay": 1e-2,
    # Training loop settings
    "max_iters": 2000,
    "eval_interval": 200,
    "eval_batches": 50,
}

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Config: n_embd={CONFIG['n_embd']}, n_heads={CONFIG['n_heads']}, "
      f"n_layers={CONFIG['n_layers']}, block_size={CONFIG['block_size']}")

## Load Artifacts

Load vocabulary and token tensors from `2. Preprocessing.ipynb` outputs for consistent splits.

In [ ]:
# Resolve project root regardless of where the notebook kernel starts.
cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"Cannot locate 'data' directory from cwd: {cwd}")

ARTIFACTS_DIR = PROJECT_ROOT / "data" / "artifacts"

# --- Load vocabulary ---
with open(ARTIFACTS_DIR / "char_vocab.json", "r", encoding="utf-8") as f:
    vocab_payload = json.load(f)

vocab_size = vocab_payload["vocab_size"]
chars = vocab_payload["chars"]
stoi = vocab_payload["stoi"]
itos = {int(k): v for k, v in vocab_payload["itos"].items()}

def encode(text: str):
    """Convert a string to a list of integer token ids."""
    return [stoi[ch] for ch in text]

def decode(indices):
    """Convert a list of integer token ids back to a string."""
    return "".join(itos[int(i)] for i in indices)

# --- Load pre-encoded tensors ---
train_ids = torch.load(ARTIFACTS_DIR / "train_ids.pt", weights_only=True)
val_ids = torch.load(ARTIFACTS_DIR / "val_ids.pt", weights_only=True)
test_ids = torch.load(ARTIFACTS_DIR / "test_ids.pt", weights_only=True)

print(f"Vocab size: {vocab_size}")
print(f"Train tokens: {len(train_ids):,} | Val tokens: {len(val_ids):,} | Test tokens: {len(test_ids):,}")

# --- Batch sampling function ---
block_size = CONFIG["block_size"]
batch_size = CONFIG["batch_size"]

def get_batch(split: str):
    """Sample a random batch of (x, y) pairs for next-token prediction."""
    data = {"train": train_ids, "val": val_ids, "test": test_ids}[split]
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# Quick verification
x, y = get_batch("train")
print(f"\nBatch shapes — x: {tuple(x.shape)}, y: {tuple(y.shape)}")
print(f"Sample decode (first 60 chars): {decode(x[0][:60].tolist())}")

## Model Architecture

We build a decoder-only GPT in four parts:

- `CausalSelfAttention`: masked multi-head attention (`scaled_dot_product_attention`)
- `FeedForward`: position-wise MLP (4x expansion + GELU)
- `TransformerBlock`: pre-norm residual attention + FFN
- `GPT`: embeddings -> blocks -> final norm -> vocab logits

Pattern used throughout: `x + sublayer(LayerNorm(x))`.

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention.

    Uses torch.nn.functional.scaled_dot_product_attention which automatically
    dispatches to FlashAttention or memory-efficient attention kernels on
    supported hardware (e.g. H100), avoiding the need for explicit causal masks.
    """

    def __init__(self, n_embd: int, n_heads: int, block_size: int, dropout: float):
        super().__init__()
        assert n_embd % n_heads == 0, "n_embd must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = n_embd // n_heads
        self.dropout = dropout

        # Combined Q, K, V projection for efficiency
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=False)
        # Output projection
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # Project to Q, K, V and split into heads: (B, T, C) -> 3x (B, n_heads, T, head_dim)
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention with causal mask (FlashAttention path)
        attn_out = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )

        # Merge heads back: (B, n_heads, T, head_dim) -> (B, T, C)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(attn_out))

### Feed-Forward Network

A 2-layer MLP per token position: expand 4x, apply GELU, project back.

$$\text{FFN}(x)=\text{GELU}(xW_1+b_1)W_2+b_2$$

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network with 4x hidden expansion and GELU."""

    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

### Transformer Block

Each block uses pre-norm residual updates:

```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

Residuals help gradient flow; LayerNorm improves stability.

In [ ]:
class TransformerBlock(nn.Module):
    """Single Transformer block: pre-norm attention + pre-norm FFN with residuals."""

    def __init__(self, n_embd: int, n_heads: int, block_size: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

### The Full GPT Model

Pipeline:
1. Token embeddings
2. Position embeddings
3. `N` Transformer blocks
4. Final LayerNorm
5. Linear LM head to logits

`generate()` performs autoregressive next-token sampling.

In [ ]:
class GPT(nn.Module):
    """
    Decoder-only GPT language model.

    Combines token/position embeddings, a stack of Transformer blocks,
    a final layer norm, and a linear head projecting to vocabulary logits.
    """

    def __init__(self, vocab_size: int, n_embd: int, n_heads: int,
                 n_layers: int, block_size: int, dropout: float):
        super().__init__()
        self.block_size = block_size

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.Sequential(*[
            TransformerBlock(n_embd, n_heads, block_size, dropout)
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying: share token embedding weights with the output projection.
        # This reduces parameters and improves generalization on small datasets.
        self.token_emb.weight = self.lm_head.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        """Initialize weights with small normal distribution; zero out biases."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        """
        Forward pass.

        Args:
            idx: (B, T) token indices
            targets: (B, T) target token indices (optional; if provided, computes loss)

        Returns:
            logits: (B, T, vocab_size)
            loss: scalar cross-entropy loss (None if targets not provided)
        """
        B, T = idx.shape
        assert T <= self.block_size, f"Sequence length {T} exceeds block_size {self.block_size}"

        tok_emb = self.token_emb(idx)                          # (B, T, n_embd)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = self.drop(tok_emb + pos_emb)                       # (B, T, n_embd)

        x = self.blocks(x)                                     # (B, T, n_embd)
        x = self.ln_f(x)                                       # (B, T, n_embd)
        logits = self.lm_head(x)                               # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0):
        """
        Autoregressive text generation.

        Crops context to block_size, predicts next-token logits,
        applies temperature scaling + softmax, and samples.

        Args:
            idx: (B, T) conditioning token indices
            max_new_tokens: number of tokens to generate
            temperature: softmax temperature (higher = more random)

        Returns:
            idx: (B, T + max_new_tokens) extended sequence
        """
        self.eval()
        for _ in range(max_new_tokens):
            # Crop context to the last block_size tokens
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            # Take logits at the last time step and apply temperature
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

### Model Instantiation

Instantiate the baseline model and inspect parameter count.

Baseline: `n_embd=128`, `n_heads=4`, `n_layers=4`.

In [ ]:
model = GPT(
    vocab_size=vocab_size,
    n_embd=CONFIG["n_embd"],
    n_heads=CONFIG["n_heads"],
    n_layers=CONFIG["n_layers"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel structure:\n{model}")

## Sanity Checks

Before full training, run three quick checks:

1. Shape check on one batch
2. Overfit one fixed batch (100 steps)
3. Baseline training run (2,000 steps)

In [ ]:
# --- Test 1: Single-batch forward pass shape verification ---
model.eval()
with torch.no_grad():
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

expected_shape = (batch_size, block_size, vocab_size)
actual_shape = tuple(logits.shape)

print(f"Expected logits shape: {expected_shape}")
print(f"Actual logits shape:   {actual_shape}")
print(f"Loss: {loss.item():.4f}")
assert actual_shape == expected_shape, f"Shape mismatch! Got {actual_shape}"
print("\n✓ Shape test PASSED — forward pass produces correct dimensions.")

### Overfitting Test

Train on one fixed batch for 100 steps.

Expected: loss drops near zero. If not, gradient flow is likely wrong.

In [ ]:
# --- Test 2: Overfit a single batch (100 iterations) ---
# Re-initialize the model to start fresh for this test.
overfit_model = GPT(
    vocab_size=vocab_size,
    n_embd=CONFIG["n_embd"],
    n_heads=CONFIG["n_heads"],
    n_layers=CONFIG["n_layers"],
    block_size=CONFIG["block_size"],
    dropout=0.0,  # disable dropout for memorization test
).to(device)

optimizer = torch.optim.AdamW(overfit_model.parameters(), lr=1e-3)

# Fix a single batch
xb_fixed, yb_fixed = get_batch("train")

overfit_model.train()
print("Overfitting single batch for 100 iterations:")
print("-" * 45)

for step in range(100):
    _, loss = overfit_model(xb_fixed, yb_fixed)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 10 == 0 or step == 99:
        print(f"  Step {step:3d} | Loss: {loss.item():.6f}")

final_loss = loss.item()
print("-" * 45)
if final_loss < 0.1:
    print(f"✓ Overfit test PASSED — final loss {final_loss:.6f} is near zero.")
else:
    print(f"⚠ Final loss {final_loss:.6f} is higher than expected. Check gradient flow.")

del overfit_model, optimizer  # free memory

### Baseline Training Run

Train on full data for 2,000 steps and evaluate periodically on validation data.

Setup:
- AdamW + weight decay
- Gradient clipping (`max_norm=1.0`)
- Validation loss averaged over 50 batches

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_batches: int = CONFIG["eval_batches"]):
    """Estimate train and val loss over multiple random batches."""
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(eval_batches):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        results[split] = np.mean(losses)
    model.train()
    return results


# --- Test 3: Baseline training loop (2,000 iterations) ---
# Re-seed for reproducibility of this training run
torch.manual_seed(SEED)

# Re-initialize the main model fresh for baseline training
model = GPT(
    vocab_size=vocab_size,
    n_embd=CONFIG["n_embd"],
    n_heads=CONFIG["n_heads"],
    n_layers=CONFIG["n_layers"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

max_iters = CONFIG["max_iters"]
eval_interval = CONFIG["eval_interval"]

print(f"Training for {max_iters} iterations (eval every {eval_interval})...")
print("=" * 60)

model.train()
for step in range(max_iters):

    # Periodic evaluation
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model)
        print(f"  Step {step:5d} | Train loss: {losses['train']:.4f} | Val loss: {losses['val']:.4f}")

    # Forward + backward
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    # Gradient clipping for training stability
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

print("=" * 60)
final_losses = estimate_loss(model)
print(f"\nBaseline complete.")
print(f"  Final train loss: {final_losses['train']:.4f}")
print(f"  Final val loss:   {final_losses['val']:.4f}")

## Text Generation

Use `generate()` for autoregressive sampling:

1. Encode a prompt (for example, `"ROMEO:"`)
2. Repeatedly predict and append the next token
3. Decode tokens back to text

`temperature < 1.0` is more deterministic; `> 1.0` is more random.

In [ ]:
# --- Generate text from a prompt ---
prompt = "ROMEO:"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

print(f"Prompt: \"{prompt}\"")
print(f"Generating 500 characters with temperature=1.0...\n")
print("=" * 60)

generated_ids = model.generate(context, max_new_tokens=500, temperature=1.0)
generated_text = decode(generated_ids[0].tolist())
print(generated_text)
print("=" * 60)

# Also show a lower-temperature (more conservative) sample
print(f"\n\nGenerating 500 characters with temperature=0.7...\n")
print("=" * 60)

generated_ids_cool = model.generate(context, max_new_tokens=500, temperature=0.7)
generated_text_cool = decode(generated_ids_cool[0].tolist())
print(generated_text_cool)
print("=" * 60)

## Summary & Next Steps

Built:
- Decoder-only character GPT in PyTorch
- FlashAttention-backed causal attention
- Pre-norm residual blocks
- Autoregressive generation with temperature

Validated:
- Correct output shape
- Single-batch overfit works
- Baseline training loss improves

Next:
- Run W&B sweeps (LR, depth, width, heads, dropout)
- Add cosine schedule + warmup
- Enable mixed precision (`torch.amp`)
- Save best checkpoint as a W&B artifact